# Section 3: Anomaly Detection for Cyber Threats

This fixed experiment cleans one unmodified CIC-IDS2018 flow file, trains Isolation Forest and an Autoencoder on benign traffic only, and selects anomaly thresholds on labelled validation data.

In [ ]:
%pip install -q joblib matplotlib numpy pandas scikit-learn torch

In [ ]:
import json
import random
import shutil
import sys
import urllib.request
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, average_precision_score, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
AUTOENCODER_EPOCHS = 12  # Change to 1 for a quick pipeline check.
COLAB = "google.colab" in sys.modules
ROOT = Path("/content/section_03_workspace") if COLAB else Path.cwd()
DATA_FILE = ROOT / "data/raw/cic-ids2018/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv"
PROCESSED_DIR = ROOT / "data/processed/section_03"
MODELS_DIR = ROOT / "models/section_03"
RESULTS_DIR = ROOT / "reports/section_03"

for directory in (DATA_FILE.parent, PROCESSED_DIR, MODELS_DIR, RESULTS_DIR / "metrics", RESULTS_DIR / "figures"):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Download the unclean CIC-IDS2018 CSV

In [ ]:
CIC_URL = "https://cse-cic-ids2018.s3.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv"
if not DATA_FILE.exists():
    request = urllib.request.Request(CIC_URL, headers={"User-Agent": "COMP70049-assignment"})
    with urllib.request.urlopen(request) as response, DATA_FILE.open("wb") as output:
        shutil.copyfileobj(response, output)

## 2. Audit, clean, and split the data

In [ ]:
TRAFFIC_FEATURES = [
    "Dst Port", "Protocol", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts", "TotLen Fwd Pkts", "TotLen Bwd Pkts",
    "Fwd Pkt Len Max", "Fwd Pkt Len Min", "Fwd Pkt Len Mean", "Fwd Pkt Len Std", "Bwd Pkt Len Max", "Bwd Pkt Len Min", "Bwd Pkt Len Mean", "Bwd Pkt Len Std",
    "Flow Byts/s", "Flow Pkts/s", "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Tot", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min", "Bwd IAT Tot", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Fwd Pkts/s", "Bwd Pkts/s", "Pkt Len Min", "Pkt Len Max", "Pkt Len Mean", "Pkt Len Std", "Pkt Len Var",
    "FIN Flag Cnt", "SYN Flag Cnt", "RST Flag Cnt", "PSH Flag Cnt", "ACK Flag Cnt", "URG Flag Cnt", "Down/Up Ratio", "Pkt Size Avg",
    "Fwd Seg Size Avg", "Bwd Seg Size Avg", "Subflow Fwd Pkts", "Subflow Fwd Byts", "Subflow Bwd Pkts", "Subflow Bwd Byts",
    "Init Fwd Win Byts", "Init Bwd Win Byts", "Fwd Act Data Pkts", "Fwd Seg Size Min",
    "Active Mean", "Active Std", "Active Max", "Active Min", "Idle Mean", "Idle Std", "Idle Max", "Idle Min", "hour", "day_of_week",
]

In [ ]:
raw = pd.read_csv(DATA_FILE, low_memory=False)
raw_profile = {
    "records": len(raw), "columns": raw.shape[1],
    "duplicate_records": int(raw.duplicated().sum()),
    "native_missing_values": int(raw.isna().sum().sum()),
}

frame = raw.copy()
frame.columns = [str(column).replace("\ufeff", "").strip() for column in frame.columns]
frame = frame.drop(columns=[column for column in frame.columns if column.lower().startswith("unnamed:")])
labels = frame["Label"].astype("string").str.strip()
missing_labels = labels.isna() | labels.eq("")
embedded_headers = labels.str.casefold().eq("label")
frame = frame.loc[~(missing_labels | embedded_headers)].copy()
duplicates_removed = int(frame.duplicated().sum())
frame = frame.drop_duplicates().reset_index(drop=True)
labels = frame["Label"].astype("string").str.strip()

timestamps = pd.to_datetime(
    frame["Timestamp"].astype("string").str.strip(), errors="coerce", format="mixed", dayfirst=True,
)
frame["hour"] = timestamps.dt.hour
frame["day_of_week"] = timestamps.dt.dayofweek

numeric = pd.DataFrame(index=frame.index)
infinities = 0
coercions = 0
for column in TRAFFIC_FEATURES:
    original = frame[column]
    converted = pd.to_numeric(original, errors="coerce")
    coercions += int((original.notna() & original.astype("string").str.strip().ne("") & converted.isna()).sum())
    infinities += int(np.isinf(converted.to_numpy(float)).sum())
    numeric[column] = converted.replace([np.inf, -np.inf], np.nan)

dropped_features = numeric.columns[numeric.isna().mean() > 0.4].tolist()
numeric = numeric.drop(columns=dropped_features)
feature_columns = numeric.columns.tolist()
cleaned = numeric.copy()
cleaned["attack_label"] = labels.to_numpy(str)
cleaned["label"] = (~labels.str.casefold().eq("benign")).astype(int).to_numpy()

cleaning_report = {
    "records_after_cleaning": len(cleaned),
    "rows_without_label_removed": int(missing_labels.sum()),
    "embedded_header_rows_removed": int(embedded_headers.sum()),
    "duplicate_records_removed": duplicates_removed,
    "infinite_values_replaced_with_missing": infinities,
    "non_numeric_values_coerced_to_missing": coercions,
    "invalid_timestamps": int(timestamps.isna().sum()),
    "features_dropped_for_missingness": dropped_features,
    "remaining_missing_values": int(numeric.isna().sum().sum()),
    "benign_records": int((cleaned["label"] == 0).sum()),
    "anomaly_records": int((cleaned["label"] == 1).sum()),
}
(PROCESSED_DIR / "data-quality-report.json").write_text(
    json.dumps({"raw_profile": raw_profile, "cleaning_report": cleaning_report}, indent=2)
)
display(pd.Series({**raw_profile, **cleaning_report}).to_frame("value"))

In [ ]:
benign = cleaned[cleaned["label"] == 0].sample(n=120000, random_state=SEED).reset_index(drop=True)
anomalies = cleaned[cleaned["label"] == 1].sample(n=18000, random_state=SEED + 1).reset_index(drop=True)

train = benign.iloc[:72000].copy()
validation = pd.concat([benign.iloc[72000:96000], anomalies.iloc[:9000]], ignore_index=True)
test = pd.concat([benign.iloc[96000:], anomalies.iloc[9000:]], ignore_index=True)
validation = validation.sample(frac=1, random_state=SEED + 2).reset_index(drop=True)
test = test.sample(frac=1, random_state=SEED + 3).reset_index(drop=True)

display(pd.DataFrame([
    {"split": name, "records": len(part), "benign": int((part["label"] == 0).sum()),
     "anomaly": int((part["label"] == 1).sum())}
    for name, part in (("train", train), ("validation", validation), ("test", test))
]))

## 3. Preprocess and evaluate anomaly scores

In [ ]:
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("variance", VarianceThreshold()),
    ("scaler", StandardScaler()),
])
train_features = preprocessor.fit_transform(train[feature_columns]).astype(np.float32)
validation_features = preprocessor.transform(validation[feature_columns]).astype(np.float32)
test_features = preprocessor.transform(test[feature_columns]).astype(np.float32)
validation_labels = validation["label"].to_numpy(np.int64)
test_labels = test["label"].to_numpy(np.int64)

feature_names = preprocessor.named_steps["imputer"].get_feature_names_out(feature_columns)
feature_names = feature_names[preprocessor.named_steps["variance"].get_support()].tolist()
(PROCESSED_DIR / "selected-features.json").write_text(json.dumps(feature_names, indent=2))
display(pd.DataFrame({"split": ["train", "validation", "test"],
                      "records": [len(train_features), len(validation_features), len(test_features)],
                      "features": [train_features.shape[1]] * 3}))

In [ ]:
def select_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return thresholds[np.argmax(f1_values)]

def evaluate(name, labels, scores, threshold, filename):
    predictions = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions).ravel()
    metrics = {
        "threshold": float(threshold),
        "true_positive_rate": float(tp / (tp + fn)),
        "false_positive_rate": float(fp / (fp + tn)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "average_precision": float(average_precision_score(labels, scores)),
        "roc_auc": float(roc_auc_score(labels, scores)),
        "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],
    }
    (RESULTS_DIR / "metrics" / f"{filename}.json").write_text(json.dumps(metrics, indent=2))

    figure, axes = plt.subplots(1, 3, figsize=(15, 4))
    ConfusionMatrixDisplay.from_predictions(
        labels, predictions, display_labels=["Benign", "Anomaly"], cmap="Blues", colorbar=False, ax=axes[0],
    )
    precision, recall, _ = precision_recall_curve(labels, scores)
    axes[1].plot(recall, precision)
    axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall curve")
    axes[2].hist(scores[labels == 0], bins=60, alpha=0.65, label="Benign")
    axes[2].hist(scores[labels == 1], bins=60, alpha=0.65, label="Anomaly")
    axes[2].axvline(threshold, color="black", linestyle="--")
    axes[2].set_title("Score distribution")
    axes[2].legend()
    figure.suptitle(name)
    figure.tight_layout()
    figure.savefig(RESULTS_DIR / "figures" / f"{filename}-evaluation.png", dpi=180)
    plt.show()
    return metrics

## 4. Isolation Forest

In [ ]:
isolation_forest = IsolationForest(
    n_estimators=200, max_samples=10000, contamination="auto", random_state=SEED, n_jobs=1,
)
isolation_forest.fit(train_features)
isolation_validation_scores = -isolation_forest.score_samples(validation_features)
isolation_threshold = select_threshold(validation_labels, isolation_validation_scores)
isolation_scores = -isolation_forest.score_samples(test_features)
isolation_metrics = evaluate(
    "Isolation Forest", test_labels, isolation_scores, isolation_threshold, "isolation-forest",
)
joblib.dump({"model": isolation_forest, "preprocessor": preprocessor,
             "threshold": isolation_threshold, "feature_names": feature_names},
            MODELS_DIR / "isolation-forest.joblib")
display(pd.Series(isolation_metrics).drop("confusion_matrix").to_frame("value"))

## 5. Dense Autoencoder

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, feature_count):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_count, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 12),
            nn.Linear(12, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, feature_count),
        )

    def forward(self, values):
        return self.network(values)

def feature_loader(features, shuffle=False):
    dataset = TensorDataset(torch.from_numpy(features))
    return DataLoader(dataset, batch_size=1024, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(SEED) if shuffle else None)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autoencoder = Autoencoder(train_features.shape[1]).to(device)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.001, weight_decay=0.00001)
loss_function = nn.MSELoss()
train_loader = feature_loader(train_features, True)
normal_validation_features = validation_features[validation_labels == 0]

def reconstruction_scores(features):
    autoencoder.eval()
    scores = []
    with torch.no_grad():
        for (batch,) in feature_loader(features):
            batch = batch.to(device)
            scores.extend(torch.mean((autoencoder(batch) - batch) ** 2, dim=1).cpu().numpy())
    return np.asarray(scores)

In [ ]:
history = []
best_validation_loss = float("inf")
for epoch in range(1, AUTOENCODER_EPOCHS + 1):
    autoencoder.train()
    total_loss = 0
    for (batch,) in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss = loss_function(autoencoder(batch), batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch)

    validation_loss = reconstruction_scores(normal_validation_features).mean()
    history.append({"epoch": epoch, "training_loss": total_loss / len(train_features),
                    "normal_validation_loss": validation_loss})
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_state = {name: value.detach().cpu().clone() for name, value in autoencoder.state_dict().items()}

autoencoder.load_state_dict(best_state)
history_frame = pd.DataFrame(history)
display(history_frame)
history_frame.set_index("epoch").plot(figsize=(7, 4), title="Autoencoder training history")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures/autoencoder-training-history.png", dpi=180)
plt.show()

autoencoder_validation_scores = reconstruction_scores(validation_features)
autoencoder_threshold = select_threshold(validation_labels, autoencoder_validation_scores)
autoencoder_scores = reconstruction_scores(test_features)
autoencoder_metrics = evaluate(
    "Autoencoder", test_labels, autoencoder_scores, autoencoder_threshold, "autoencoder",
)
autoencoder_metrics.update({"best_normal_validation_loss": float(best_validation_loss),
                            "epochs": AUTOENCODER_EPOCHS, "device": str(device)})
(RESULTS_DIR / "metrics/autoencoder.json").write_text(json.dumps(autoencoder_metrics, indent=2))
torch.save({"model_state": best_state, "threshold": autoencoder_threshold,
            "feature_names": feature_names}, MODELS_DIR / "autoencoder.pt")
display(pd.Series(autoencoder_metrics).drop("confusion_matrix").to_frame("value"))

## 6. Compare and export

In [ ]:
metric_names = ["true_positive_rate", "false_positive_rate", "precision", "f1", "average_precision", "roc_auc"]
comparison = pd.DataFrame([
    {"model": "Isolation Forest", **{metric: isolation_metrics[metric] for metric in metric_names}},
    {"model": "Autoencoder", **{metric: autoencoder_metrics[metric] for metric in metric_names}},
])
comparison.to_csv(RESULTS_DIR / "model-comparison.csv", index=False)
(RESULTS_DIR / "run-summary.json").write_text(json.dumps({
    "train_records": len(train), "validation_records": len(validation),
    "test_records": len(test), "features": train_features.shape[1],
    "autoencoder_epochs": AUTOENCODER_EPOCHS,
}, indent=2))
display(comparison.style.format({metric: "{:.4f}" for metric in metric_names}))

if COLAB:
    export_dir = ROOT / "section_03_export"
    shutil.copytree(RESULTS_DIR, export_dir / "reports", dirs_exist_ok=True)
    shutil.copytree(MODELS_DIR, export_dir / "models", dirs_exist_ok=True)
    shutil.make_archive("/content/section_03_results", "zip", root_dir=export_dir)

## Interpretation

- High true-positive rate is not useful when the false-positive rate is also high.
- Average precision and ROC AUC show whether scores rank anomalies above benign flows.
- Training uses benign records only; labels are used only for threshold selection and final evaluation.
- Statistical outliers are retained because they may be the attacks the models should detect.
- Results from one CIC-IDS2018 day do not establish deployment readiness.